In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch

In [2]:
!curl -L -o wine-quality-dataset.zip\
  https://www.kaggle.com/api/v1/datasets/download/yasserh/wine-quality-dataset

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 21984  100 21984    0     0  58456      0 --:--:-- --:--:-- --:--:-- 58456


In [3]:
!unzip /content/wine-quality-dataset.zip

Archive:  /content/wine-quality-dataset.zip
  inflating: WineQT.csv              


In [4]:
df = pd.read_csv('/content/WineQT.csv')
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,0
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,2
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,3
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1138,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6,1592
1139,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6,1593
1140,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5,1594
1141,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6,1595


In [5]:
df['quality'].value_counts()

,count
quality,
5,483
6,462
7,143
4,33
8,16
3,6


In [6]:
df.drop(['Id'], axis=1, inplace=True)

In [7]:
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
1,7.8,0.880,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5
2,7.8,0.760,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5
3,11.2,0.280,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6
4,7.4,0.700,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1138,6.3,0.510,0.13,2.3,0.076,29.0,40.0,0.99574,3.42,0.75,11.0,6
1139,6.8,0.620,0.08,1.9,0.068,28.0,38.0,0.99651,3.42,0.82,9.5,6
1140,6.2,0.600,0.08,2.0,0.090,32.0,44.0,0.99490,3.45,0.58,10.5,5
1141,5.9,0.550,0.10,2.2,0.062,39.0,51.0,0.99512,3.52,0.76,11.2,6


In [8]:
X = df.drop(['quality'], axis=1)
y = df['quality'].copy()

In [9]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2)

In [10]:
X_train.shape, X_valid.shape, X_test.shape

((731, 11), (183, 11), (229, 11))

In [11]:
y_train.shape, y_valid.shape, y_test.shape

((731,), (183,), (229,))

In [12]:
wine_imputer = SimpleImputer(strategy="median")
wine_scaler = StandardScaler()

In [13]:
X_train_transfomer = wine_scaler.fit_transform(X_train)
X_valid_transfomer = wine_scaler.transform(X_valid)
X_test_transfomer = wine_scaler.transform(X_test)

In [14]:
y_min = y.min() # 3-u çıxartmaq üçün

In [15]:
y_train = y_train.to_numpy() - y_min
y_valid = y_valid.to_numpy() - y_min
y_test = y_test.to_numpy() - y_min

In [16]:
train_tensor = TensorDataset(torch.FloatTensor(X_train_transfomer), torch.LongTensor(y_train))
valid_tensor = TensorDataset(torch.FloatTensor(X_valid_transfomer), torch.LongTensor(y_valid))
test_tensor  = TensorDataset(torch.FloatTensor(X_test_transfomer),  torch.LongTensor(y_test))

In [17]:
train_loader = DataLoader(train_tensor, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_tensor, batch_size=32)
test_loader = DataLoader(test_tensor, batch_size=32)

In [18]:
input_size = X_train_transfomer.shape[1]

model = nn.Sequential(
    nn.Linear(input_size, 1024),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 64),
    nn.ReLU(),
    nn.Linear(64, 6)
)

In [23]:
device='cuda'

In [24]:
def train_model(model, criterion, optimizer, train_loader, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            # Ensure y_batch is long and squeezed for CrossEntropyLoss
            loss = criterion(y_pred, y_batch.long().squeeze())
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        print(f"Epoch: {epoch+1}/{n_epochs}, loss: {total_loss:.4f}")

In [25]:
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [26]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

Sequential(
  (0): Linear(in_features=11, out_features=1024, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.3, inplace=False)
  (3): Linear(in_features=1024, out_features=256, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.3, inplace=False)
  (6): Linear(in_features=256, out_features=64, bias=True)
  (7): ReLU()
  (8): Linear(in_features=64, out_features=6, bias=True)
)

In [27]:
train_model(model, xentropy, optimizer, train_loader, 100)

Epoch: 1/100, loss: 4.5339
Epoch: 2/100, loss: 3.6646
Epoch: 3/100, loss: 2.9547
Epoch: 4/100, loss: 3.0957
Epoch: 5/100, loss: 3.4441
Epoch: 6/100, loss: 3.5287
Epoch: 7/100, loss: 3.5420
Epoch: 8/100, loss: 3.6545
Epoch: 9/100, loss: 3.5486
Epoch: 10/100, loss: 2.8684
Epoch: 11/100, loss: 3.1998
Epoch: 12/100, loss: 3.5453
Epoch: 13/100, loss: 3.0337
Epoch: 14/100, loss: 3.0492
Epoch: 15/100, loss: 2.6922
Epoch: 16/100, loss: 2.6322
Epoch: 17/100, loss: 3.0559
Epoch: 18/100, loss: 2.6605
Epoch: 19/100, loss: 3.2277
Epoch: 20/100, loss: 2.6559
Epoch: 21/100, loss: 2.7106
Epoch: 22/100, loss: 2.1910
Epoch: 23/100, loss: 3.0992
Epoch: 24/100, loss: 3.3835
Epoch: 25/100, loss: 2.6784
Epoch: 26/100, loss: 2.3399
Epoch: 27/100, loss: 2.3830
Epoch: 28/100, loss: 2.5078
Epoch: 29/100, loss: 1.9525
Epoch: 30/100, loss: 3.2617
Epoch: 31/100, loss: 3.4000
Epoch: 32/100, loss: 1.6758
Epoch: 33/100, loss: 3.0637
Epoch: 34/100, loss: 3.0004
Epoch: 35/100, loss: 2.6616
Epoch: 36/100, loss: 2.1489
E